# Optimization 
We consider the optimization problem  
$$
\min_{\theta \in \mathbb{R}^d} \; f(\theta),
$$
where $f$ is differentiable and $\nabla f(\theta)$ denotes its gradient.  
All gradient-based methods update the parameter as  
$$
\theta_{t+1} = \theta_t - \eta_t \, g_t,
$$
where $g_t$ is an estimator of the gradient and $\eta_t > 0$ is the learning rate.

---

## 1. **Batch Gradient Descent (BGD)**

Uses the **full dataset** to compute the gradient at each step:
$$
g_t = \nabla_{\theta} \frac{1}{N}\sum_{i=1}^{N} L(f(x_i;\theta_t), y_i).
$$

- **Update rule:**
  $$
  \theta_{t+1} = \theta_t - \eta_t \nabla f(\theta_t)
  $$
- **Pros:** Smooth convergence, accurate gradient.  
- **Cons:** Computationally expensive for large $N$.

---

## 2. **Stochastic Gradient Descent (SGD)**

Approximates the true gradient using a **single sample** (or very few):
$$
g_t = \nabla_{\theta} L(f(x_i;\theta_t), y_i).
$$

- **Update rule:**
  $$
  \theta_{t+1} = \theta_t - \eta_t g_t
  $$
- **Pros:** Fast updates, works well for large-scale or online data.  
- **Cons:** High variance in updates, noisy convergence.

---

## 3. **Mini-Batch Gradient Descent**

Uses a **subset (mini-batch)** of data $\mathcal{B}_t \subset \{1,\ldots,N\}$:
$$
g_t = \frac{1}{|\mathcal{B}_t|} \sum_{i \in \mathcal{B}_t} \nabla_{\theta} L(f(x_i;\theta_t), y_i).
$$

- Balances efficiency and stability.  
- The most commonly used in deep learning.

---

## 4. **RMSProp**

Modifies Adagrad by using an **exponential moving average**:
$$
\begin{aligned}
E[g^2]_t &= \beta E[g^2]_{t-1} + (1 - \beta) g_t^2, \\
\theta_{t+1} &= \theta_t - \frac{\eta}{\sqrt{E[g^2]_t + \epsilon}} \, g_t.
\end{aligned}
$$

- Keeps learning rate stable over time.  
- Standard choice for recurrent networks.

---

## 5. **Adam (Adaptive Moment Estimation)**

Combines **Momentum** and **RMSProp**:
$$
\begin{aligned}
m_t &= \beta_1 m_{t-1} + (1-\beta_1) g_t, \\
v_t &= \beta_2 v_{t-1} + (1-\beta_2) g_t^2, \\
\hat{m}_t &= \frac{m_t}{1 - \beta_1^t}, \quad
\hat{v}_t = \frac{v_t}{1 - \beta_2^t}, \\
\theta_{t+1} &= \theta_t - \frac{\eta \, \hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}.
\end{aligned}
$$

- **Pros:** Fast, robust, adaptive per-parameter learning.  
- **Cons:** May converge to suboptimal minima; often requires learning-rate tuning.

---

## 6. **AdamW (Weight Decay Adam)**

A variant of Adam with **decoupled weight decay**:
$$
\theta_{t+1} = (1 - \eta \lambda) \theta_t - \frac{\eta \, \hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}.
$$

- Regularization by explicit weight decay $\lambda$.  
- Commonly used in modern deep networks (e.g., Transformers).

---

## 7. **Summary Table**

| Method | Typical Use |
|:-------|:-------------|
| BGD | Small datasets |
| SGD | Online learning |
| Mini-batch GD | Deep learning |
| RMSProp | RNNs, non-stationary data |
| Adam | Default optimizer |
| AdamW | Transformer training |


In [ ]:
# 🔧 Common Gradient Descent Optimizers in PyTorch
# ------------------------------------------------
# This example shows how you can switch between optimizers
# simply by changing a few characters.

import torch
import torch.nn as nn
import torch.optim as optim

# Example: simple fnn model
class FNN(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=32, output_dim=1):
        super(FNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return self.network(x)

# Example learning rate
lr = 0.01

# Select one optimizer by uncommenting its line

# --- Basic gradient descent variants ---
# optimizer = optim.SGD(model.parameters(), lr=lr)              # SGD (vanilla)
# optimizer = optim.RMSprop(model.parameters(), lr=lr)           # RMSProp
# optimizer = optim.Adam(model.parameters(), lr=lr)              # Adam
# optimizer = optim.AdamW(model.parameters(), lr=lr)             # AdamW (with decoupled weight decay)

# --- Optional: specify weight decay (L2 regularization) ---
# optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

# Training loop
#for epoch in range(3):
    # forward pass
    #x = torch.randn(16, 10)
    #y = torch.randn(16, 1)
    loss = nn.MSELoss()(model(x), y)

    # backward pass
    #optimizer.zero_grad()
    #loss.backward()
    #optimizer.step()

    #print(f"Epoch {epoch+1}: loss = {loss.item():.4f}")

# Grid Search
## Overview
In training neural networks, the **choice of hyperparameters** — such as the learning rate, hidden layer size, and optimizer — greatly influences model performance.  
A **Grid Search** systematically explores combinations of these hyperparameters to identify the configuration that minimizes the model loss or maximizes validation accuracy.

---

## General Idea

We aim to solve
$$
\min_{\lambda \in \Lambda} \; \mathcal{L}_{\text{val}}(\hat{\theta}(\lambda)),
$$
where:
- $\lambda$ represents a **set of hyperparameters** (e.g. learning rate, hidden units, optimizer type),
- $\hat{\theta}(\lambda)$ are the trained model parameters under configuration $\lambda$,
- $\mathcal{L}_{\text{val}}$ is the validation loss.

---

## Key Steps

1. **Define the model structure**  
   Build a simple Feedforward Neural Network (FNN) using PyTorch’s `nn.Module`.

2. **Specify the hyperparameter grid**  
   Example:
   ```python
   param_grid = {
       'hidden_dim': [16, 32],
       'lr': [0.01, 0.001],
       'optimizer': ['SGD', 'Adam', 'RMSprop']
   }
## Generate all combinations

Using `itertools.product()` to create the full grid:
$$
\Lambda \;=\; \{\, (h, \eta, \text{opt}) \;:\; h \in H,\; \eta \in \mathrm{LR},\; \text{opt} \in \mathrm{OPT} \,\}.
$$

---

## Train and evaluate each configuration

For each $(h, \eta, \text{opt})$, train the FNN briefly and record the final loss.

---

## Select the best configuration

Choose
$$
\lambda^\star \;=\; \arg\min_{\lambda \in \Lambda} \; \mathcal{L}(\lambda).
$$

---

## Code Logic Summary

| Step | Description |
|:--:|:--|
| 1️⃣ | Define FNN architecture (`FNN` class) |
| 2️⃣ | Generate hyperparameter combinations with `itertools.product` |
| 3️⃣ | Instantiate model and optimizer dynamically |
| 4️⃣ | Train for a few epochs and compute loss |
| 5️⃣ | Record and print configuration with lowest loss |

---

## Why It Matters

- **Systematic Exploration:** ensures fair comparison between different settings.  
- **Reproducible:** results can be logged and replicated easily.  
- **Extensible:** add new parameters (activation, batch size, etc.) with minimal code changes.

---

## Practical Tip

In practice:
- Use a **training/validation split** (or cross-validation).
- Run **multiple epochs** per configuration.
- Consider scalable tools:
  - `sklearn.model_selection.GridSearchCV`
  - `skorch` *(scikit-learn interface for PyTorch)*
  - `ray[tune]` or `optuna` *(for large-scale / Bayesian optimization)*


In [ ]:
# 🔍 Grid Search for Feedforward Neural Network (FNN)
# ---------------------------------------------------
# This example shows how to perform a simple grid search
# over hyperparameters (optimizer, learning rate, hidden size)
# using PyTorch.

import torch
import torch.nn as nn
import torch.optim as optim
from itertools import product

# 1️⃣ Define FNN model
class FNN(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=32, output_dim=1):
        super(FNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.network(x)

# 2️⃣ Dummy data (for illustration)
x_train = torch.randn(100, 10)
y_train = torch.randn(100, 1)

# 3️⃣ Hyperparameter grid
param_grid = {
    'hidden_dim': [16, 32],
    'lr': [0.01, 0.001],
    'optimizer': ['SGD', 'Adam', 'RMSprop']
}

# Create all possible combinations
grid = list(product(param_grid['hidden_dim'], param_grid['lr'], param_grid['optimizer']))

best_loss = float('inf')
best_params = None

# 4️⃣ Grid Search Loop
for hidden_dim, lr, opt_name in grid:
    model = FNN(input_dim=10, hidden_dim=hidden_dim, output_dim=1)
    criterion = nn.MSELoss()

    # Dynamically select optimizer
    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr)
    elif opt_name == 'RMSprop':
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    elif opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Simple one-epoch training (for demonstration)
    for epoch in range(3):
        pred = model(x_train)
        loss = criterion(pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Record best configuration
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_params = (hidden_dim, lr, opt_name)

    print(f"Tested: hidden={hidden_dim}, lr={lr}, opt={opt_name}, loss={loss.item():.4f}")

print(f"\n✅ Best config → hidden={best_params[0]}, lr={best_params[1]}, optimizer={best_params[2]}")


# Reading assignments
The Essential Main Ideas of Neural Networks: https://www.youtube.com/watch?v=CqOfi41LfDw&list=PLblh5JKOoLUIxGDQs4LFFD--41Vzf-ME1&index=2
<br>The Chain Rule, Clearly Explained!!!: https://www.youtube.com/watch?v=wl1myxrtQHQ&list=PLblh5JKOoLUIxGDQs4LFFD--41Vzf-ME1&index=3
<br>Gradient Descent, Step-by-Step: https://www.youtube.com/watch?v=sDv4f4s2SB8&list=PLblh5JKOoLUIxGDQs4LFFD--41Vzf-ME1&index=4
<br>Neural Networks Pt. 3: ReLU In Action!!!:,https://www.youtube.com/watch?v=68BZ5f7P94E&list=PLblh5JKOoLUIxGDQs4LFFD--41Vzf-ME1&index=8
<br>Neural Networks Pt. 4: Multiple Inputs and Outputs
: https://www.youtube.com/watch?v=83LYR-1IcjA&list=PLblh5JKOoLUIxGDQs4LFFD--41Vzf-ME1&index=9